# Exercise 3: Two CSTRs in Series

## System overview

**Given parameters:**
- Flow rate: $F = 100$ L/h
- Feed substrate concentration: $S_0 = 10$ g/L
- Yield coefficient: $Y_{X/S} = 0.5$ g-cell / g-substrate
- Monod parameters: $\mu_m = 1$ h⁻¹, $K_S = 0.75$ g/L
- Maintenance energy neglected
- Four volume configurations (V1 + V2 = 1000 L total):
  - (a) V1 = 800 L, V2 = 200 L
  - (b) V1 = 200 L, V2 = 800 L
  - (c) V1 = 900 L, V2 = 100 L
  - (d) V1 = 100 L, V2 = 900 L

## Model Derivation

### Monod Growth Rate
$$\mu = \frac{\mu_m S}{K_S + S}$$

### Reactor 1: Steady-State Mass Balances
The dilution rate is $D_1 = F / V_1$.

**Cell balance** (feed is sterile, $X_0 = 0$):
$$0 = -D_1 X_1 + \mu_1 X_1 \implies \mu_1 = D_1$$

From Monod: $\mu_1 = D_1 \implies S_1 = \frac{K_S D_1}{\mu_m - D_1}$

**Substrate balance:**
$$0 = D_1(S_0 - S_1) - \frac{\mu_1 X_1}{Y_{X/S}}$$
$$\implies X_1 = Y_{X/S}(S_0 - S_1)$$

### Reactor 2: Steady-State Mass Balances
The dilution rate is $D_2 = F / V_2$. The feed to reactor 2 comes from reactor 1 ($S_1, X_1$).

**Cell balance:**
$$0 = D_2(X_1 - X_2) + \mu_2 X_2$$
$$\implies \mu_2 = D_2 \left(1 - \frac{X_1}{X_2}\right)$$

**Substrate balance:**
$$0 = D_2(S_1 - S_2) - \frac{\mu_2 X_2}{Y_{X/S}}$$
$$\implies X_2 = X_1 + Y_{X/S}(S_1 - S_2)$$

The two equations for reactor 2 (cell + substrate) combined with Monod form a nonlinear system solved numerically.

In [7]:
import numpy as np
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Parameters ────────────────────────────────────────────────────────────────
F      = 100.0   # L/h  – volumetric flow rate
S0     = 10.0    # g/L  – feed substrate concentration
Yxs    = 0.5     # g-cell / g-substrate
mu_m   = 1.0     # h⁻¹  – max specific growth rate
Ks     = 0.75    # g/L  – saturation constant

# Monod growth rate
def mu(S):
    return mu_m * S / (Ks + S)

print("Parameters loaded successfully.")
print(f"  F={F} L/h | S0={S0} g/L | Yxs={Yxs} | mu_m={mu_m} h⁻¹ | Ks={Ks} g/L")

Parameters loaded successfully.
  F=100.0 L/h | S0=10.0 g/L | Yxs=0.5 | mu_m=1.0 h⁻¹ | Ks=0.75 g/L


## Reactor 1: Analytical Solution

Because the feed is sterile ($X_0 = 0$), the steady-state cell balance in reactor 1 gives $\mu_1 = D_1$ directly.

$$S_1 = \frac{K_S D_1}{\mu_m - D_1}, \qquad X_1 = Y_{X/S}(S_0 - S_1)$$

> **Washout condition:** If $D_1 \geq \mu_m$ there is no stable steady state (washout). The model returns $X_1 = 0, S_1 = S_0$ in that case.

In [ ]:
def solve_reactor1(V1):

    D1 = F / V1
    if D1 >= mu_m:
        # Washout
        return S0, 0.0, 0.0, D1
    S1  = Ks * D1 / (mu_m - D1)
    X1  = Yxs * (S0 - S1)
    mu1 = mu(S1)   # should equal D1
    return S1, X1, mu1, D1

## Reactor 2: Numerical Solution

The feed to reactor 2 is $(S_1, X_1)$. The unknowns are $S_2$ and $X_2$.

**System of equations to solve:**

$$f_1 = D_2(S_1 - S_2) - \frac{\mu(S_2)\, X_2}{Y_{X/S}} = 0$$

$$f_2 = D_2(X_1 - X_2) + \mu(S_2)\, X_2 = 0$$

In [ ]:
def solve_reactor2(V2, S1, X1):

    D2 = F / V2

    def equations(vars):
        S2, X2 = vars
        mu2 = mu(S2)
        eq1 = D2 * (S1 - S2) - mu2 * X2 / Yxs   # substrate balance
        eq2 = D2 * (X1 - X2) + mu2 * X2          # cell balance
        return [eq1, eq2]

    # Initial guess: slight decrease from reactor-1 values
    x0 = [max(S1 * 0.5, 1e-6), X1 * 1.1]
    sol = fsolve(equations, x0, full_output=True)
    S2, X2 = sol[0]

    # Clamp to physical range
    S2 = max(S2, 0.0)
    X2 = max(X2, 0.0)
    mu2 = mu(S2)
    return S2, X2, mu2, D2

## Solve All Configurations (a)–(d)

In [10]:
configurations = [
    ('(a)', 800, 200),
    ('(b)', 200, 800),
    ('(c)', 900, 100),
    ('(d)', 100, 900),
]

results = []

print(f"{'Config':>8} | {'V1 (L)':>7} | {'V2 (L)':>7} | "
      f"{'S1 (g/L)':>10} | {'X1 (g/L)':>10} | {'mu1 (h⁻¹)':>10} | "
      f"{'S2 (g/L)':>10} | {'X2 (g/L)':>10} | {'mu2 (h⁻¹)':>10}")
print('-' * 105)

for label, V1, V2 in configurations:
    S1, X1, mu1, D1 = solve_reactor1(V1)
    S2, X2, mu2, D2 = solve_reactor2(V2, S1, X1)
    results.append(dict(label=label, V1=V1, V2=V2,
                        D1=D1, S1=S1, X1=X1, mu1=mu1,
                        D2=D2, S2=S2, X2=X2, mu2=mu2))
    print(f"{label:>8} | {V1:>7} | {V2:>7} | "
          f"{S1:>10.4f} | {X1:>10.4f} | {mu1:>10.4f} | "
          f"{S2:>10.4f} | {X2:>10.4f} | {mu2:>10.4f}")

  Config |  V1 (L) |  V2 (L) |   S1 (g/L) |   X1 (g/L) |  mu1 (h⁻¹) |   S2 (g/L) |   X2 (g/L) |  mu2 (h⁻¹)
---------------------------------------------------------------------------------------------------------
     (a) |     800 |     200 |     0.1071 |     4.9464 |     0.1250 |     0.0039 |     4.9981 |     0.0052
     (b) |     200 |     800 |     0.7500 |     4.6250 |     0.5000 |     0.0070 |     4.9965 |     0.0093
     (c) |     900 |     100 |     0.0938 |     4.9531 |     0.1111 |     0.0066 |     4.9967 |     0.0087
     (d) |     100 |     900 |    10.0000 |     0.0000 |     0.0000 |    10.0000 |     0.0000 |     0.9302


## Single 1000 L Reactor (Reference Case)

For comparison we compute the steady state for a single CSTR of 1000 L.

In [11]:
V_single = 1000.0
S_single, X_single, mu_single, D_single = solve_reactor1(V_single)

print("Single 1000 L reactor:")
print(f"  D      = {D_single:.4f} h⁻¹")
print(f"  S_out  = {S_single:.4f} g/L")
print(f"  X_out  = {X_single:.4f} g/L")
print(f"  mu_out = {mu_single:.4f} h⁻¹")

Single 1000 L reactor:
  D      = 0.1000 h⁻¹
  S_out  = 0.0833 g/L
  X_out  = 4.9583 g/L
  mu_out = 0.1000 h⁻¹


## Summary Table

| Config |  V1  |  V2  |  D1   |   S1    |   X1    |  μ1   |  D2   |   S2    |   X2    |  μ2   |  ΔS    |  ΔX    |
|:------:|-----:|-----:|------:|--------:|--------:|------:|------:|--------:|--------:|------:|-------:|-------:|
| (a)    |  800 |  200 | 0.125 | 0.1071  | 4.9464  | 0.1250| 0.500 | 0.0039  | 4.9981  | 0.0052| 0.1032 | 0.0516 |
| (b)    |  200 |  800 | 0.500 | 0.7500  | 4.6250  | 0.5000| 0.125 | 0.0070  | 4.9965  | 0.0093| 0.7430 | 0.3715 |
| (c)    |  900 |  100 | 0.111 | 0.0938  | 4.9531  | 0.1111| 1.000 | 0.0066  | 4.9967  | 0.0087| 0.0872 | 0.0436 |
| (d)    |  100 |  900 | 1.000 |10.0000  | 0.0000  | 0.0000| 0.111 |10.0000  | 0.0000  | 0.9302| 0.0000 | 0.0000 |


| Config |  V1  | V2 |  D1   |   S1    | X1 | μ1 | D2 |   S2    |   X2    |  μ2   | ΔS | ΔX |
|:------:|-----:|:--:|------:|--------:|:--:|:--:|:--:|--------:|--------:|------:|:--:|:--:|
| Single | 1000 | —  | 0.100 | 0.0833  | —  | —  | —  | 0.0833  | 4.9583  | 0.1000| —  | —  |


**Units:**  
- V [L]  
- D [h⁻¹]  
- S [g/L]  
- X [g/L]  
- μ [h⁻¹]